# Chapter 5 – Solutions
**Questions answered:** Q1 (bias–variance / minimum-variance combination) and Q6 (bootstrap vs. GLM standard errors on the **Default** data set).

## Question 1
> Using basic statistical properties of the variance, as well as single-variable calculus, derive (5.6). In other words, prove that $\alpha$ given by (5.6) does indeed minimize $\mathrm{Var}(\alpha X + (1-\alpha)Y)$.

### Setup
Let $\sigma_X^2 = \mathrm{Var}(X)$, $\sigma_Y^2 = \mathrm{Var}(Y)$, and $\sigma_{XY} = \mathrm{Cov}(X,Y)$. For a combined variable $Z = \alpha X + (1-\alpha) Y$, using the bilinearity of covariance:

$$
f(\alpha) = \mathrm{Var}(\alpha X + (1-\alpha)Y) = \alpha^2 \sigma_X^2 + (1-\alpha)^2 \sigma_Y^2 + 2\alpha(1-\alpha)\sigma_{XY}
$$

### Step 1 — First-order condition
Differentiate $f(\alpha)$ with respect to $\alpha$ and set it to zero:

$$
f'(\alpha) = 2\alpha \sigma_X^2 - 2(1-\alpha)\sigma_Y^2 + 2\sigma_{XY}(1-2\alpha) = 0
$$

Expanding:

$$
\alpha \sigma_X^2 - \sigma_Y^2 + \alpha \sigma_Y^2 + \sigma_{XY} - 2\alpha \sigma_{XY} = 0
$$

Collecting the terms in $\alpha$:

$$
\alpha\left(\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}\right) = \sigma_Y^2 - \sigma_{XY}
$$

$$
\boxed{\alpha = \dfrac{\sigma_Y^2 - \sigma_{XY}}{\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}}} \qquad \text{(this is exactly equation (5.6))}
$$

### Step 2 — Second-order condition (confirming it's a *minimum*)

$$
f''(\alpha) = 2\sigma_X^2 + 2\sigma_Y^2 - 4\sigma_{XY} = 2\left(\sigma_X^2 + \sigma_Y^2 - 2\sigma_{XY}\right) = 2\,\mathrm{Var}(X-Y)
$$

Since a variance is always $\ge 0$, $f''(\alpha) \ge 0$ everywhere (and $>0$ as long as $X-Y$ is not a constant, i.e. $X$ and $Y$ are not perfectly correlated with equal variance). A function with a single stationary point and non-negative second derivative there is convex, so that stationary point is a **global minimum**, not a maximum or saddle point.

Hence $\alpha$ from (5.6) minimizes $\mathrm{Var}(\alpha X + (1-\alpha)Y)$. $\blacksquare$

### Numerical / symbolic check
We can confirm the algebra above symbolically with **sympy**, and also verify it numerically for a random covariance pair.

In [8]:
import sympy as sp

alpha, sx2, sy2, sxy = sp.symbols('alpha sigma_X^2 sigma_Y^2 sigma_XY', real=True)

# Var(alpha*X + (1-alpha)*Y)
f = alpha**2*sx2 + (1-alpha)**2*sy2 + 2*alpha*(1-alpha)*sxy

# First derivative and solve for alpha
fprime = sp.diff(f, alpha)
alpha_star = sp.solve(sp.Eq(fprime, 0), alpha)[0]
alpha_star_simplified = sp.simplify(alpha_star)
print('alpha* =', alpha_star_simplified)

# Compare to formula (5.6)
formula_56 = (sy2 - sxy) / (sx2 + sy2 - 2*sxy)
print('Matches (5.6)? ', sp.simplify(alpha_star_simplified - formula_56) == 0)

# Second derivative (should be positive => minimum)
fdoubleprime = sp.diff(f, alpha, 2)
print('f\'\'(alpha) =', fdoubleprime, ' (constant, non-negative since it is 2*Var(X-Y))')

alpha* = (-sigma_XY + sigma_Y^2)/(-2*sigma_XY + sigma_X^2 + sigma_Y^2)
Matches (5.6)?  True
f''(alpha) = 2*(-2*sigma_XY + sigma_X^2 + sigma_Y^2)  (constant, non-negative since it is 2*Var(X-Y))


In [12]:

import numpy as np

# Var(alpha*X + (1-alpha)*Y), then compare with formula (5.6)

rng = np.random.default_rng(0)
n = 100_000

X = rng.normal(0, 2.0, n)
Y = 0.6 * X + rng.normal(0, 1.5, n)  # induces correlation between X and Y

sigma_X2 = np.var(X, ddof=1)
sigma_Y2 = np.var(Y, ddof=1)
sigma_XY = np.cov(X, Y, ddof=1)[0, 1]

# Formula (5.6)
alpha_formula = (
    (sigma_Y2 - sigma_XY)
    / (sigma_X2 + sigma_Y2 - 2 * sigma_XY)
)
# Brute-force grid search
alphas = np.linspace(0, 1, 1001)
variances = [
    np.var(a * X + (1 - a) * Y, ddof=1)
    for a in alphas
]
alpha_brute = alphas[np.argmin(variances)]
print(f"alpha from formula (5.6): {alpha_formula:.4f}")
print(f"alpha from brute-force grid search: {alpha_brute:.4f}")

alpha from formula (5.6): 0.4492
alpha from brute-force grid search: 0.4490


## Question 6
> We continue to consider the use of a logistic regression model to predict the probability of **default** using **income** and **balance** on the **Default** data set. In particular, we will now compute estimates for the standard errors of the **income** and **balance** logistic regression coefficients in two different ways: (1) using the bootstrap, and (2) using the standard formula for computing the standard errors in the **sm.GLM()** function. Do not forget to set a random seed before beginning your analysis.

(a) Using the **summarize()** and **sm.GLM()** functions, determine the estimated standard errors for the coefficients associated with **income** and **balance** in a multiple logistic regression model that uses both predictors.

(b) Write a function, **boot_fn()**, that takes as input the **Default** data set as well as an index of the observations, and that outputs the coefficient estimates for **income** and **balance** in the multiple logistic regression model.

(c) Following the bootstrap example in the lab, use your **boot_fn()** function to estimate the standard errors of the logistic regression coefficients for **income** and **balance**.

(d) Comment on the estimated standard errors obtained using the **sm.GLM()** function and using the bootstrap.

In [17]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from ISLP import load_data

# Set a random seed before beginning the analysis
rng = np.random.default_rng(1)

Default = load_data('Default')
Default['default01'] = (Default['default'] == 'Yes').astype(int)
Default.head()

,default,student,balance,income,default01
0,No,No,729.526495,44361.625074,0
1,No,Yes,817.180407,12106.134700,0
2,No,No,1073.549164,31767.138947,0
3,No,No,529.250605,35704.493935,0
4,No,No,785.655883,38463.495879,0


### (a) Standard errors from **sm.GLM()**
Fit the multiple logistic regression of **default** on **income** and **balance**, and read off the standard errors reported by the model summary (these come from the inverse Fisher information / Hessian of the log-likelihood — the "standard formula").

In [20]:
X = sm.add_constant(Default[['income', 'balance']])
y = Default['default01']

glm_model = sm.GLM(y, X, family=sm.families.Binomial())
glm_results = glm_model.fit()
print(glm_results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:              default01   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Sat, 29 Aug 2026   Deviance:                       1579.0
Time:                        19:57:00   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.0

In [22]:
glm_se = glm_results.bse[['income', 'balance']]
print('Standard errors from sm.GLM():')
print(glm_se)

Standard errors from sm.GLM():
income     0.000005
balance    0.000227
dtype: float64


### (b) **boot_fn()**
**boot_fn()** takes the full **Default** data set and a vector of row indices (with replacement, when called from the bootstrap loop), refits the logistic regression on that resampled data, and returns the fitted coefficients for **income** and **balance**.

In [27]:
def boot_fn(data, index):
    """
    Fit the logistic regression of default01 ~ income + balance
    on data.iloc[index], and return the (income, balance) coefficients.
    """
    d = data.iloc[index]
    X_ = sm.add_constant(d[['income', 'balance']], has_constant='add')
    y_ = d['default01']
    model_ = sm.GLM(y_, X_, family=sm.families.Binomial()).fit()
    return model_.params[['income', 'balance']].values

# Sanity check: passing the full, unresampled index should reproduce the fit from part (a)
full_index = np.arange(len(Default))
print('boot_fn on full sample:', boot_fn(Default, full_index))
print('sm.GLM() coefficients :', glm_results.params[['income', 'balance']].values)

boot_fn on full sample: [2.08089755e-05 5.64710295e-03]
sm.GLM() coefficients : [2.08089755e-05 5.64710295e-03]


### (c) Bootstrap standard errors
Following the ISLR lab pattern, we write a generic **boot()** routine that repeatedly draws a bootstrap sample of row indices (sampling **n** rows **with replacement** from the **n** original rows), applies **boot_fn()** to each resampled data set, and then reports the standard deviation of each coefficient across the `R` bootstrap replicates as the bootstrap estimate of its standard error.

In [30]:
def boot(data, func, R, rng):
    """
    Generic bootstrap routine.
    data : DataFrame to resample from
    func : function(data, index) -> 1D array of statistic(s)
    R    : number of bootstrap replicates
    rng  : numpy random Generator (for reproducibility)
    Returns the array of bootstrap estimates (R x p) and their standard errors (length p).
    """
    n = len(data)
    first = func(data, np.arange(n))
    estimates = np.zeros((R, len(first)))
    for i in range(R):
        index = rng.integers(0, n, size=n)  # sample n indices with replacement
        estimates[i, :] = func(data, index)
    se = estimates.std(axis=0, ddof=1)
    return estimates, se

R = 1000
boot_estimates, boot_se = boot(Default, boot_fn, R, rng)

boot_se = pd.Series(boot_se, index=['income', 'balance'])
print(f'Bootstrap standard errors (R = {R} replicates):')
print(boot_se)

Bootstrap standard errors (R = 1000 replicates):
income     0.000005
balance    0.000226
dtype: float64


In [31]:
comparison = pd.DataFrame({
    'sm.GLM() SE': glm_se,
    'Bootstrap SE': boot_se
})
comparison['Difference'] = comparison['Bootstrap SE'] - comparison['sm.GLM() SE']
comparison

,sm.GLM() SE,Bootstrap SE,Difference
income,0.000005,0.000005,-1.116426e-07
balance,0.000227,0.000226,-1.468628e-06


### (d) Comment

The standard errors produced by **sm.GLM()** and by the bootstrap are **very close** to one another for both **income** and **balance** — typically agreeing to within a small fraction of their magnitude (the exact numbers will vary slightly with the random seed and number of bootstrap replicates **R**, but the two approaches consistently agree).

This agreement makes sense conceptually:

- The **sm.GLM()** standard errors are the **"formula" (model-based) standard errors**. They are derived from the asymptotic theory of maximum likelihood estimation: under the assumed logistic regression model, the MLE is asymptotically normal with a covariance matrix equal to the inverse of the (expected) Fisher information matrix. These standard errors rely on the model being correctly specified (i.e., the true relationship between **default**, **income**, and **balance** really is logistic).
- The **bootstrap standard errors** make no such distributional assumption. They are obtained empirically, by refitting the model on many resampled versions of the data and looking directly at the spread of the resulting coefficient estimates. The bootstrap is therefore more robust in principle, since it does not rely on the correctness of the model's asymptotic formula-it only relies on the original sample being representative of the population.

Because the two sets of standard errors come out so similar here, this is evidence that the logistic regression model's assumptions (and the asymptotic approximation used by **sm.GLM()**) are reasonable for this data set-with **n = 10{,}000** observations, the sample is large enough that the asymptotic (formula-based) standard errors and the empirical (bootstrap) standard errors have essentially converged to the same answer. If the model were misspecified, or the sample size were much smaller, we would expect to see a larger discrepancy between the two.